In [2]:
import pandas as pd
import json

# Load program enrollment data
df = pd.read_csv("ProgramTypeCountbyWrittenLanguage.csv")

# Load NYC ZIP code boundaries
with open("nyc-zip-code-tabulation-areas-polygons.geojson", "r") as f:
    nyc_geojson = json.load(f)

# Extract valid NYC ZIP codes
nyc_zip_codes = {
    str(feature["properties"]["postalCode"]).zfill(5)
    for feature in nyc_geojson["features"]
}

# Clean ZIP codes in enrollment data
df["Zip_Code"] = (
    df["Zip_Code"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

# Filter to NYC ZIP codes
nyc_df = df[df["Zip_Code"].isin(nyc_zip_codes)].copy()

# Check results
print("Original rows:", len(df))
print("NYC rows:", len(nyc_df))
print("Rows removed:", len(df) - len(nyc_df))
print("Unique NYC ZIP codes:", nyc_df["Zip_Code"].nunique())

# Save filtered dataset
nyc_df.to_csv(
    "ProgramTypeCountbyWrittenLanguage_NYC.csv",
    index=False
)

# Preview
display(nyc_df.head())

Original rows: 23853
NYC rows: 6499
Rows removed: 17354
Unique NYC ZIP codes: 222


,Zip_Code,Program_Type,Preferred_Written_Language,Enrollments_Count
5,10001,Medicaid,ARABIC,<50
6,10001,Essential Plan,ARABIC,<50
7,10001,Medicaid,CHINESE,177
8,10001,CHP,CHINESE,<50
9,10001,QHP APTC/CSR,CHINESE,<50


In [3]:

import pandas as pd
import numpy as np


df = pd.read_csv(
    "ProgramTypeCountbyWrittenLanguage_NYC.csv",
    dtype={"Zip_Code": "string"}
)

df.columns = df.columns.str.strip()

print("Available columns:")
print(df.columns.tolist())


df["Zip_Code"] = (
    df["Zip_Code"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

df["Program_Type"] = (
    df["Program_Type"]
    .astype("string")
    .str.strip()
)


df["Enrollments_Count"] = pd.to_numeric(
    df["Enrollments_Count"]
    .astype(str)
    .str.replace(",", "", regex=False),
    errors="coerce"
).fillna(0)


program_types = [
    "Medicaid",
    "CHP",
    "QHP APTC w/o CSR",
    "QHP Full Pay",
    "Essential Plan"
]

#

program_pivot = df.pivot_table(
    index="Zip_Code",
    columns="Program_Type",
    values="Enrollments_Count",
    aggfunc="sum",
    fill_value=0
)

program_pivot = program_pivot.reindex(
    columns=program_types,
    fill_value=0
)


program_pivot["Total_Enrollments"] = (
    program_pivot[program_types].sum(axis=1)
)



for program in program_types:

    program_pivot[f"{program} Percentage"] = (
        program_pivot[program]
        / program_pivot["Total_Enrollments"].replace(0, np.nan)
        * 100
    ).fillna(0).round(2)


program_pivot = program_pivot.rename(
    columns={
        program: f"{program} Count"
        for program in program_types
    }
)


ordered_columns = ["Total_Enrollments"]

for program in program_types:
    ordered_columns.extend([
        f"{program} Count",
        f"{program} Percentage"
    ])

program_pivot = program_pivot[ordered_columns]


program_zip_df = (
    program_pivot
    .reset_index()
    .sort_values("Zip_Code")
)


percentage_columns = [
    f"{program} Percentage"
    for program in program_types
]

program_zip_df["Percentage_Total"] = (
    program_zip_df[percentage_columns].sum(axis=1)
)

print(
    "Percentage totals (should be approximately 100 "
    "for ZIP codes with enrollments):"
)

print(
    program_zip_df[
        ["Zip_Code", "Percentage_Total"]
    ].head(10)
)

program_zip_df = program_zip_df.drop(
    columns="Percentage_Total"
)


program_zip_df.to_csv(
    "NYC_Program_Enrollment_by_ZIP.csv",
    index=False
)

print("Original number of rows:", len(df))
print("Unique NYC ZIP codes:", len(program_zip_df))

display(program_zip_df.head(300))


Available columns:
['Zip_Code', 'Program_Type', 'Preferred_Written_Language', 'Enrollments_Count']
Percentage totals (should be approximately 100 for ZIP codes with enrollments):
Program_Type Zip_Code  Percentage_Total
0               10001            100.01
1               10002            100.00
2               10003            100.01
3               10004            100.00
4               10005            100.00
5               10006            100.00
6               10007            100.00
7               10009            100.00
8               10010            100.00
9               10011            100.00
Original number of rows: 6499
Unique NYC ZIP codes: 222


Program_Type,Zip_Code,Total_Enrollments,Medicaid Count,Medicaid Percentage,CHP Count,CHP Percentage,QHP APTC w/o CSR Count,QHP APTC w/o CSR Percentage,QHP Full Pay Count,QHP Full Pay Percentage,Essential Plan Count,Essential Plan Percentage
0,10001,6119.0,4711.0,76.99,104.0,1.70,112.0,1.83,313.0,5.12,879.0,14.37
1,10002,31922.0,24466.0,76.64,739.0,2.32,421.0,1.32,413.0,1.29,5883.0,18.43
2,10003,5382.0,3478.0,64.62,102.0,1.90,278.0,5.17,739.0,13.73,785.0,14.59
3,10004,342.0,225.0,65.79,0.0,0.00,0.0,0.00,57.0,16.67,60.0,17.54
4,10005,800.0,524.0,65.50,0.0,0.00,0.0,0.00,116.0,14.50,160.0,20.00
...,...,...,...,...,...,...,...,...,...,...,...,...
217,11691,28610.0,22284.0,77.89,2215.0,7.74,245.0,0.86,114.0,0.40,3752.0,13.11
218,11692,7513.0,6303.0,83.89,251.0,3.34,0.0,0.00,0.0,0.00,959.0,12.76
219,11693,3448.0,2999.0,86.98,86.0,2.49,0.0,0.00,0.0,0.00,363.0,10.53
220,11694,3267.0,2489.0,76.19,157.0,4.81,64.0,1.96,92.0,2.82,465.0,14.23
